In [ ]:
# ==============================================================================
# WHAT THIS DOES:
#   - Installs pyabsa
#   - Patches a known pyabsa dependency bug: metric_visualizer's
#     UpdateChecker.check() has a signature mismatch that throws a
#     TypeError on import. This patch no-ops that check before pyabsa
#     ever triggers it. This is a library bug, not anything wrong with
#     your code.
#   - Confirms Python, GPU, and PyABSA are all working.
# ==============================================================================

get_ipython().system('pip install -q -U pyabsa')

import update_checker
update_checker.UpdateChecker.check = lambda *args, **kwargs: None

import sys
import torch
import pyabsa
from pyabsa import AspectPolarityClassification as APC

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version     : {sys.version.split()[0]}")
print(f"PyABSA version     : {pyabsa.__version__}")

cuda_ok = torch.cuda.is_available()
print(f"CUDA available     : {cuda_ok}")
if cuda_ok:
    print(f"GPU device         : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory (GB)    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > "
          "set Hardware accelerator to GPU, then Runtime > Restart session, "
          "then re-run this cell.")

print("APC module         : loaded OK")
print("=" * 60)
print("Setup complete.")
print("=" * 60)


In [ ]:
get_ipython().system('wget -q "https://zenodo.org/records/18500517/files/EduRABSA_Dataset.zip?download=1" -O /content/EduRABSA_Dataset.zip')
get_ipython().system('unzip -q -o /content/EduRABSA_Dataset.zip -d /content/EduRABSA_Dataset')
get_ipython().system('find /content/EduRABSA_Dataset -maxdepth 4 -name "*.json"')


In [ ]:
import json
import random
import subprocess
from pathlib import Path

random.seed(42)  # fixed seed: split is reproducible if you ever need to redo it

JSON_DIR = Path("/content/EduRABSA_Dataset/EduRABSA_Dataset/2_annotated_dataset_files/Annotation_raw_output")
TRAIN_VAL_JSON = JSON_DIR / "Train_validation_Merged_ASQE_N4000.json"
DEV_TEST_JSON = JSON_DIR / "DEV_TEST_Merged_ASQE_N2500.json"

OUTPUT_DIR = Path("/content/apc_dataset")
VALID_FRACTION_OF_DEVTEST = 0.70  # 70% valid, 30% test

VALID_SENTIMENTS = {"Positive", "Negative", "Neutral"}


def load_reviews(filepath):
    """Load a review JSON file. Returns dict of {review_id: review_dict}."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def review_to_apc_blocks(review):
    """Convert one review's Quadruplets into PyABSA APC 3-line blocks."""
    text = review["text"]
    blocks = []

    for quad in review["Quadruplets"]:
        aspect = quad["aspect"]
        sentiment = quad["sentiment"]

        if aspect is None or str(aspect).strip().lower() == "null":
            continue  # implicit aspect, no literal span to mask
        if sentiment not in VALID_SENTIMENTS:
            continue  # not a usable 3-class label

        idx = text.find(aspect)
        if idx == -1:
            continue  # aspect text doesn't literally appear in the review

        masked_sentence = text[:idx] + "$T$" + text[idx + len(aspect):]
        blocks.append((masked_sentence, aspect, sentiment))

    return blocks


def build_review_blocks(review_dict):
    """Returns [(review_id, blocks), ...] for every review with at least one usable block."""
    review_blocks = []
    empty_count = 0
    for review_id, review in review_dict.items():
        blocks = review_to_apc_blocks(review)
        if blocks:
            review_blocks.append((review_id, blocks))
        else:
            empty_count += 1
    if empty_count:
        print(f"  NOTE: {empty_count} reviews produced zero usable APC blocks "
              f"and were excluded entirely.")
    return review_blocks


def write_blocks(filepath, review_blocks):
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        for review_id, blocks in review_blocks:
            for sentence, aspect, label in blocks:
                f.write(sentence + "\n")
                f.write(aspect + "\n")
                f.write(label + "\n")


def report(name, review_blocks, source_total):
    n_examples = sum(len(b) for _, b in review_blocks)
    print(f"{name:8s}: {len(review_blocks):5d} reviews used "
          f"(of {source_total} source reviews), {n_examples:5d} aspect-level examples")


# --- Train set: 100% of the train_val pool ---
print("Loading train_val JSON...")
train_val_data = load_reviews(TRAIN_VAL_JSON)
print(f"  Source reviews in file: {len(train_val_data)}")
train_review_blocks = build_review_blocks(train_val_data)
write_blocks(OUTPUT_DIR / "train" / "train.txt", train_review_blocks)
report("Train", train_review_blocks, len(train_val_data))

# --- Valid/Test: 70/30 split of the dev_test pool, by whole review ---
print("\nLoading dev_test JSON...")
dev_test_data = load_reviews(DEV_TEST_JSON)
print(f"  Source reviews in file: {len(dev_test_data)}")
devtest_review_blocks = build_review_blocks(dev_test_data)

shuffled = devtest_review_blocks.copy()
random.shuffle(shuffled)

split_point = int(len(shuffled) * VALID_FRACTION_OF_DEVTEST)
valid_review_blocks = shuffled[:split_point]
test_review_blocks = shuffled[split_point:]

write_blocks(OUTPUT_DIR / "valid" / "valid.txt", valid_review_blocks)
write_blocks(OUTPUT_DIR / "test" / "test.txt", test_review_blocks)
report("Valid", valid_review_blocks, len(dev_test_data))
report("Test", test_review_blocks, len(dev_test_data))

print()
print("Sanity check -- review counts should match the source JSON closely:")
print(f"  Train review count          : {len(train_review_blocks)}  (source: {len(train_val_data)})")
print(f"  Valid+Test review count     : {len(valid_review_blocks) + len(test_review_blocks)}  (source: {len(dev_test_data)})")

print()
print("Done. Folder structure:")
print(subprocess.run(["find", str(OUTPUT_DIR), "-type", "f"], capture_output=True, text=True).stdout)